# argmax-accuracy-eval — ex1: top-1 classification accuracy from logits

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `argmax-accuracy-eval`. Running the final beacon cell reports progress against the `Eval: argmax accuracy` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Eval: argmax accuracy` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`argmax-accuracy-eval`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "argmax-accuracy-eval"
DD_SUBTOPIC = "Eval: argmax accuracy"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `(logits.argmax(dim=-1) == labels).float().mean()` — quick refresher

Top-1 classification accuracy in three idioms:

```
preds   = logits.argmax(dim=-1)             # (B,) predicted class
correct = (preds == labels)                 # (B,) boolean
acc     = correct.float().mean()            # scalar in [0, 1]
```

**Why `argmax(dim=-1)`.** `logits` is `(B, C)`. We want the index of the largest logit ALONG THE CLASS AXIS for each example. `dim=-1` is the class axis regardless of whether there are extra leading dims (e.g. `(B, T, C)` for token-level outputs).

**Why `.float().mean()` and not `.sum() / len(labels)`.** Boolean tensors can't be `.mean()`-ed directly. Casting to float gives `1.0 / 0.0` per example, and `.mean()` then handles partial-last-batch sizes correctly when accumulated across batches with a weighted average.

**Logits vs probabilities — argmax is the same.** Softmax is monotonic, so `argmax(logits) == argmax(softmax(logits))`. You can skip the softmax for accuracy; the predicted class is identical.

### Exercise 1 — top-1 classification accuracy from logits

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the `(logits.argmax(dim=-1) == labels).float().mean()` accuracy pattern, including the dim=-1 axis choice and the boolean-to-float cast.
> Keywords: accuracy, argmax, eval-metric, classification
> ```

**KCs targeted:** `argmax-along-class-dim-minus-1`, `boolean-tensor-float-mean-accuracy`

Implement `ex1_top1_accuracy(logits, labels)`. The standard classification eval metric.

1. Compute `preds = logits.argmax(dim=-1)` — shape `(B,)`.
2. Compute `correct = (preds == labels)` — shape `(B,)`, dtype bool.
3. Return `correct.float().mean()` — a scalar tensor in `[0, 1]`.

Inputs:
- `logits`: `(B, C)` float tensor.
- `labels`: `(B,)` int tensor with class indices in `[0, C)`.

Output: scalar accuracy tensor.

**Critical:** use `dim=-1` (the class axis) not `dim=0` or `dim=1` explicitly — `dim=-1` correctly handles `(B, C)` and `(B, T, C)` (token-level) shapes alike. The test verifies you got this right by passing both shapes.

In [ ]:
def ex1_top1_accuracy(logits: Tensor, labels: Tensor) -> Tensor:
    """Top-1 classification accuracy: argmax along class axis, mean over examples."""
    raise NotImplementedError()


def _test_ex1():
    # === Perfect accuracy: argmax aligned with labels ===
    logits = t.tensor([
        [10.0, 0.0, 0.0],   # argmax → 0
        [0.0, 10.0, 0.0],   # argmax → 1
        [0.0, 0.0, 10.0],   # argmax → 2
        [10.0, 0.0, 0.0],   # argmax → 0
    ])
    labels = t.tensor([0, 1, 2, 0], dtype=t.long)
    acc = ex1_top1_accuracy(logits, labels)
    assert isinstance(acc, t.Tensor), f'must return a tensor, got {type(acc)}'
    assert acc.ndim == 0, f'must return a scalar, got shape {tuple(acc.shape)}'
    assert acc.dtype.is_floating_point, (
        f'must return a float tensor (cast bool→float before mean); got dtype {acc.dtype}'
    )
    assert acc.item() == 1.0, f'all predictions correct → acc=1.0; got {acc.item()}'

    # === Zero accuracy: every prediction wrong ===
    labels_wrong = t.tensor([2, 2, 0, 2], dtype=t.long)   # 0/4 correct
    acc_wrong = ex1_top1_accuracy(logits, labels_wrong)
    assert acc_wrong.item() == 0.0, f'all wrong → acc=0.0; got {acc_wrong.item()}'

    # === Half right ===
    labels_half = t.tensor([0, 1, 0, 2], dtype=t.long)   # 2/4 correct
    acc_half = ex1_top1_accuracy(logits, labels_half)
    assert abs(acc_half.item() - 0.5) < 1e-7, f'2/4 correct → acc=0.5; got {acc_half.item()}'

    # === Bound check on random data ===
    rng = t.Generator().manual_seed(13)
    B, C = 100, 10
    random_logits = t.randn(B, C, generator=rng)
    random_labels = t.randint(0, C, (B,), generator=rng)
    acc_rand = ex1_top1_accuracy(random_logits, random_labels)
    assert 0.0 <= acc_rand.item() <= 1.0, f'acc must be in [0,1]; got {acc_rand.item()}'
    # Random predictions on 10-class labels: expected acc ≈ 0.1 (loose bound).
    assert acc_rand.item() < 0.3, f'random preds should give ~0.1 acc; got {acc_rand.item()} (>0.3 suspicious)'

    # === dim=-1 handles (B, T, C) token-level shapes too ===
    # This is what catches dim=0 or dim=1 mistakes.
    tok_logits = t.tensor([
        [[10.0, 0.0], [0.0, 10.0]],   # batch 0: token 0 → class 0, token 1 → class 1
        [[0.0, 10.0], [10.0, 0.0]],   # batch 1: token 0 → class 1, token 1 → class 0
    ])  # shape (B=2, T=2, C=2)
    tok_labels = t.tensor([
        [0, 1],
        [1, 0],
    ], dtype=t.long)  # shape (B=2, T=2)
    tok_acc = ex1_top1_accuracy(tok_logits, tok_labels)
    assert tok_acc.item() == 1.0, (
        f'token-level (B,T,C) inputs should give 1.0 (all correct); got {tok_acc.item()}. '
        f'Make sure you used dim=-1 (class axis), not dim=0 or dim=1.'
    )

    # === Verify intermediate tensor types are sensible (catches forgotten .float()) ===
    # If you returned a Long mean it would error or round; we check the result is non-integer typically.
    acc_check = ex1_top1_accuracy(
        t.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 0.0]]),
        t.tensor([0, 1, 1], dtype=t.long),   # 2/3 correct → 0.6667
    )
    assert abs(acc_check.item() - 2/3) < 1e-6, (
        f'2/3 correct should give ~0.6667; got {acc_check.item()}; '
        f'did you forget the .float() cast before .mean()?'
    )
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_top1_accuracy(logits, labels):
    preds = logits.argmax(dim=-1)
    return (preds == labels).float().mean()
```

**Why bool → float → mean.** `t.tensor([True, False]).mean()` raises `RuntimeError: Can only calculate the mean of floating types`. The fix is `(preds == labels).float()` before `.mean()`. A common almost-right form is `correct.sum() / correct.numel()` — works but is less idiomatic and slightly slower (two ops vs one).

**Why argmax is enough — no softmax needed.** Softmax is MONOTONIC: if `logits[a] > logits[b]` then `softmax(logits)[a] > softmax(logits)[b]`. So `argmax(logits) == argmax(softmax(logits))`. Doing the softmax first is harmless but wasteful.

**Where this lives in real training code.** Inside `validate()` you accumulate accuracy over the val loader, weighted by batch size (just like loss). For top-K accuracy you replace `argmax(dim=-1)` with `topk(k, dim=-1).indices` and check whether `labels` is in that set.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()